# Flet I: Basics

Flet enables developers to easily build **multi-platform apps** in Python (realtime web, mobile and desktop) with no frontend experience required. The basic UI elements or widgets in Flet are called **controls** which are based on [Flutter](https://flutter.dev/). Controls are designed to follow best UI practices and have sensible defaults, so applications looks good and polished by default with minimal styling effort during development.

## Hello, world!

We create a minimal app for showing random translations of "Hello, world!"™ as follows:

```bash
mkdir flet-basics && cd flet-basics
uv init --python=3.9
uv venv
uv add "flet[all]"
uv run flet create
```

This will have the following folder structure:

```bash
.
├── README.md
├── pyproject.toml
└── src
    ├── assets
    │   ├── icon.png
    │   └── splash_android.png
    ├── flet_basics
    │   └── __init__.py
    └── main.py
```

:::{.callout-note}
The original `pyproject.toml` created by `uv init` will be replaced with the one from Flet app template.
You may have to modify the values depending on your project's needs.

:::

Then, we code the main file as follows:

```{.python filename="src/main.py"}
import flet as ft
import time
from random import randint


hello_world = [
    "Hello, world!",
    "¡Hola, mundo!",
    "Bonjour, monde !",
    "Hallo, Welt!",
    "Ciao, mondo!",
    "Olá, mundo!",
    "こんにちは、世界！",
    "안녕하세요, 세계!",
    "你好，世界！",
    "مرحباً، يا عالم!",
]


def main(page: ft.Page):    # <1>
    default = hello_world[0]
    greeting = ft.Text(default, size=60, data=default)    #<2>
    n = len(hello_world)
    
    def roll_greeting(e):   # <3>
        # Force update rolling animation
        greeting.value = ""
        page.update()
        time.sleep(0.2)
        
        # Force update final result
        greeting.data = hello_world[randint(0, n - 1)]
        greeting.value = str(greeting.data)
        page.update()


    page.floating_action_button = ft.FloatingActionButton(  # <4>
        content=ft.Icon(ft.Icons.CASINO, size=60),
        on_click=roll_greeting,
        height=60, width=60
    )

    page.add(   # <5>
        ft.Container(
            greeting,
            alignment=ft.Alignment.CENTER,
            expand=True
        )
    )


if __name__ == "__main__":
    ft.run(main)
```

1. The `main` function is what Flet calls to build the page during `ft.app`. This takes a `page` variable which is updated inside the function.
2. Here we encounter our first control `Text` which we initialize with the default string `"Hello, world!"`. Note that a control distinguishes between **data** (internal to the program) and **value** (UI-facing).
3. We define a button **callback** for the event `e` (a button click) used below. This defines the behavior of the button click. Note that we explicitly update the page after showing a transitory animation state `""` after a click (so that a click is unambiguously indicated visually).
4. Assigning a floating action button with the above callback to the page. Note the use of dice icon to style the button which is available in the [Icons library](https://flet.dev/docs/reference/icons/).
5. The counter is added within a container[^container] to configure alignment. Here `expand=True` forces the container to fill the page so centering works vertically and horizontally.

[^container]: Container also implements features like padding, margin, background color, border, width & height, clipping, shape, alignment relative to the container box.

Then, run the app using `flet run` which automatically detects[^path] the `src/main` module:

[^path]: Assuming you have `path = "src"` in the `[tool.flet.app]` section of `pyproject.toml`.

<video
  src="./img/flet/greeting.mp4"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

:::{.callout-tip}
More control for running the app:
```bash
flet run --web app.py                 # web browser on random TCP port
flet run --web --port 8000 app.py     # web browser on port 8000
flet run -d app.py                    # hot reload current dir changes
flet run -d -r app.py                 # hot reload current dir changes + all subdirs
```

:::

## Imperative UI approach

Observe that the UI is created step-by-step with layout and behavior specified. This highlights the **imperative** style of Flet
for building UIs. To handle state changes, widget properties are directly mutated in-place, then manual calls to `page.update()` are made
to force the UI to refresh. Moreover, event handlers directly manipulate the UI. This allows arbitrary Python code to be interspersed between Flet components,
and makes it very easy to track program behavior and design for simple apps.

:::{.callout-note}
This also means that it may be difficult to create complex or large apps 
since app logic, state, and UI live in the same place.
For example, it may be tricky to react to state changes in response to, say, data changes in the backend instead of direct events like clicks.
See [Declarative UI in Flet](https://flet.dev/blog/introducing-declarative-ui-in-flet) in Flet 1.0 which introduces a declarative approach alongside the existing imperative API. 
The docs describe the declarative approach succintly as `UI = f(state)` where our code `f` describes how the UI should look like for a given state, not how to build or update it.

:::

## Flet controls

As mentioned above, the UI is made of **controls** (aka widgets) with Page as the top-most control. Controls are nested into each other and can be represented as a tree with Page as the root node. 
This picture is consistent with the imperative approach described above that we used above. Note that controls are just 
Python classes and so we can create one using their constructors:

In [7]:
import flet as ft
issubclass(ft.Text, ft.Control) 

True

### Rows

Controls are added to a page using `page.add`. Alternatively, for a control `t`, one can do `page.controls.append(t)`  followed by `page.update` to display the control in the page. We already saw how to update the UI by updating the `value` of a control and then updating the page. Some controls contain other controls:

```python
page.add(
    ft.Row(controls=[
        ft.Text("A"),
        ft.Text("B"),
        ft.Text("C")
    ])
)
```

The control `ft.Row` will arrange controls together in a row. 

### Controls list

Note that one can even dynamically remove a control from the controls list.

```python
async def main(page: ft.Page):
    page.add(
        ft.Row(controls=[
            ft.Text("A", size=60),
            ft.Text("B", size=60),
            ft.Text("C", size=60)
        ])
    )

    timer = ft.Text(size=30)
    page.add(timer)

    while True:
        timer.data = 3
        for _ in range(3):
            timer.value = f"Removing controls in {timer.data} seconds..."
            timer.data -= 1
            page.update()
            await asyncio.sleep(1)

        page.controls[0].controls.pop()
        page.update()
        
        if len(page.controls[0].controls) == 0:
            timer.value = "No more controls!"
            page.update()
            break
```

<video
  src="./img/flet/controls.mp4"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

### Buttons and text fields

The following is the common pattern of a text field with hint and button:

In [ ]:
async def main(page):
    async def add_clicked(e):                                 
        page.add(ft.Checkbox(label=new_task.value))         # <1>
        new_task.value = "" # <2>
        await new_task.focus()    # <3>
        new_task.update()   

    new_task = ft.TextField(hint_text="What's needs to be done?", width=300)
    submit_button = ft.ElevatedButton("Submit", on_click=add_clicked)
    page.add(ft.Row([new_task, submit_button]))

1. A submit event creates a checkbox below (page refreshes).
2. This is followed by clearing the field. Focus allows the user to not have to click the field again by having the keyboard caret in the `new_task` control -- this assumes the user typically adds multiple tasks. Finally, only the `new_task` control updates[^page_update].
3. `<control>.focus()` has to always be awaited.

Note that the UI first builds with the the text field and submit button at the start. Then, the default in `ft.TextField` is cleared after clicking the text field which is sensible. Once the user clicks submit, the program updates the page and the text field: (1) page is reloaded with the new task checkbox appended, (2) text field value is set to blank and the keyboard focus us marked. Once these are set, the text field then updates.

[^page_update]: The page updates at the start in `page.add`.

<video
  src="./img/flet/text-submit.mp4"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

:::{.callout-note}
**Event handlers** are arbitrary functions that can modify any control in the program, as well as state variables (values and data). It can also refresh any control or the entire page.

:::

### Visible / disabled flags

Every control can be disabled using its `.disable` Boolean attribute. 
Though these are typically applied to data entry controls.
Note that the attribute can 
also be set in the constructor. Similarly, there is a `.visible` flag which
essentially prevents the controls from being rendered in the UI. Note that these 
flags are propagated recursively to every children of these controls.

```{.python filename="src/flags.py"}
import flet as ft

def main(page):
    def add_clicked(e):
        page.add(ft.Checkbox(label=new_task.value))
        new_task.value = ""
        new_task.update()

    new_task = ft.TextField(hint_text="What's needs to be done?", width=300)
    submit_button = ft.Button("Submit", on_click=add_clicked)
    
    new_task.disabled = True
    submit_button.disabled = True

    def make_visible_clicked(e):
        invisible_field.visible = True
        invisible_field.update()

    invisible_field = ft.TextField(hint_text="Invisible field", width=300)
    invisible_field.visible = False
    make_visible_button = ft.Button("Show", on_click=make_visible_clicked)
    
    page.add(ft.Row([new_task, submit_button]))
    page.add(ft.Column([invisible_field, make_visible_button]))


if __name__ == "__main__":
    ft.run(main)
```

:::{.callout-tip}
A useful trick is to set the flags as some conditional of state variables, e.g. `visible=(count > 0)`.

:::

Observe that the button shifts left when the text field is hidden, then shifts right:

<video
  src="./img/flet/flags.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

### Dropdown

Dropdowns are implemented by passing a list of options which are `ft.dropdown.Option` objects. 
The first element of an `Option` is the `key`. Note that the option's `key` is assigned as the 
dropdown's `value` when you select it.

```{.python filename="src/dropdown.py"}
import flet as ft

def main(page: ft.Page):
    DEFAULT_GRAY = "#3B3B3B"
    bg_colors = {
        "Red":   "#FF0000",
        "Green": "#00FF00",
        "Blue":  "#0000FF",
    }
    
    square = ft.Container(
        width=45,
        height=45,
        bgcolor=DEFAULT_GRAY
    )

    def button_clicked(e):
        square.bgcolor = bg_colors.get(dropdown.value, DEFAULT_GRAY)
        page.update()

    output_text = ft.Text()
    submit_button = ft.Button("Submit", on_click=button_clicked)
    dropdown = ft.Dropdown(
        width=150,
        options=[
            ft.dropdown.Option(key="Red"),
            ft.dropdown.Option(key="Green"),
            ft.dropdown.Option(key="Blue"),
        ],
    )

    page.add(
        ft.Row([dropdown, square]), 
        submit_button, 
        output_text
    )

if __name__ == "__main__":
    ft.run(main)
```

<video
  src="./img/flet/dropdown.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

## Custom controls

Since controls are classes, we can subclass them to get modified behavior. A common use-case is to create a reusable **restyled** control. Note that we can include other attributes of the original class (e.g. the event handler `on_click`) in the constructor that we will need later. The following sets background color to orange and text color in the button to green:

```python
class MyButton(ft.Button):
    def __init__(self, text, on_click=None):
        super().__init__()
        self.bgcolor = ft.Colors.ORANGE_300
        self.color = ft.Colors.GREEN_800
        self.text = text
        self.on_click = on_click
```

### Composite controls

These inherit from container controls such as `Column`, `Row`, and `Stack` to combine multiple Flet controls.
The following implements a **task item** as a row control with edit button. A text field and save button appears 
when the edit button is clicked. This example also shows how to refactor a program to make it easy to read and 
make the main function simpler.

```{.python filename="src/composite.py"}
import flet as ft

class Task(ft.Row):
    def __init__(self, text):
        super().__init__()
        self.text_view = ft.Text(text)
        self.edit_icon = ft.IconButton(icon=ft.Icons.EDIT, on_click=self.edit)
        self.text_edit = ft.TextField(text, visible=False)
        self.save_icon = ft.IconButton(visible=False, icon=ft.Icons.SAVE, on_click=self.save)
        self.controls = [       # <1>
            ft.Checkbox(),
            self.text_view, 
            self.edit_icon,   # <2>   
            self.text_edit, 
            self.save_icon,   # <3>
        ]

    def edit(self, e):
        self.text_view.visible = False
        self.edit_icon.visible = False
        self.text_edit.visible = True
        self.save_icon.visible = True
        self.update()

    def save(self, e):
        self.text_view.visible = True
        self.edit_icon.visible = True
        self.text_edit.visible = False
        self.save_icon.visible = False
        self.text_view.value   = self.text_edit.value
        self.update()

    def is_isolated(self):  # <4>
        return True


def main(page: ft.Page):
    page.add(
        Task(text="Do laundry"),
        Task(text="Cook dinner"),
    )


if __name__ == "__main__":
    ft.run(main)
```

1. A task item consists of a **row** of [checkbox]{.underline}, [text]{.underline}, [edit button]{.underline}, [edit field]{.underline}, and a [save button]{.underline}.
2. Edit button hides the first two and shows the last two. This can be thought of as **edit mode.** 
3. This is reversed with save and in addition the text field receives the value 
of the edit field value. Here we are back to **view mode** (default).
4. See [isolated controls](/courses/app-dev/01-flet.html#isolated-controls) in the next section.

:::{.callout-tip}
Thinking of **state phases** of the application can be helpful (e.g. view vs. edit mode) when designing or understanding UI code.

:::

<video
  src="./img/flet/composite.mp4"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

### Isolated controls

`Task` defined above is a custom control that contains five children:

```python
Task (Row)
 ├── Checkbox
 ├── Text (view mode)
 ├── TextField (edit mode)
 ├── IconButton (edit button)
 └── IconButton (save button)
```

When edit or save button is clicked, the task: 1) hides some children, 
2) shows other children, 3), changes values, 4) then calls `self.update()`. 
In Flet, all updates are ultimately reconciled against the page's control tree by computing a diff. However, calling `self.update()` scopes the diff to that control's subtree rather than triggering a full page-level reconciliation.

If the page later calls `update()` (for example, due to other state changes), Flet will still traverse the page tree, but controls marked as isolated are treated as [opaque leaves]{.underline} and their children are not re-walked. Repeated tree walks and overlapping mutation responsibilities can introduce unnecessary overhead and increase the risk of fragile update ordering in complex applications.

For this reason, controls that mutate their own internal structure and call `self.update()` should be **isolated** from parent reconciliation. This is done by defining an `is_isolated()` method that returns `True`. Isolation establishes a clear ownership boundary: the control becomes responsible for keeping its internal subtree consistent, while parent updates no longer attempt to reconcile that subtree. This is a performance and correctness convention rather than a hard safety guarantee, you have to ensure that the states of the child controls are consistent when the parent control (i.e. the control which defines `is_isolated`).

## Navigation and routing

Flet makes **single page applications** (SPA) with [virtual pages]{.underline} called **views** to allow organizing UI/UX. Navigation between pages are done via the URL which reflects the current state of the application. First, the **page route** is simply the part that follows the base URL (e.g. `/` for the home page and `/admin` for `https:localhost/admin`[^route]). Then, this can be used to control which **view** of the app to show the user. 

Flet provides convenient features such as automatic synchronization with browser history (so that browser back 🡄 and forward buttons works 🡆), as well programmatic control over the routes by allowing an update of `page.route` anywhere in the program. Finally, we can easily intercept the app **view pop** (which we represent using 🔙) with the `page.on_view_pop` handler. For example, we can control where the page will go via `page.go(<route>)` whenever 🔙 is clicked, or do some state modification along with.

[^route]: It follows that `/admin` is different from `/admin/`, i.e. the latter can be thought of as like home page within `/admin`.

### Page routes

```python
import flet as ft

def main(page: ft.Page):
    def route_change(e: ft.RouteChangeEvent):           # <1>
        page.add(ft.Text(f"New route: {e.route}"))

    page.on_route_change = route_change                 # <2>
    page.add(ft.Text(f"Initial route: {page.route}"))    

if __name__ == "__main__":
    ft.run(main)
```

1. We access the page route using `e.route` in a route change event `e`.
2. Assign the function to the page route event handler `on_route_change` of the page.

<video
  src="./img/flet/navigation.mp4"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

Note that all routes start with `/` and that Flet [automatically syncs]{.underline} with the [browser history]{.underline} which can be seen by the browser 🡄 moving to the left of the **linear history**, while 🡆 moves to the right of the history. In the above demo, the first request does not trigger a text field add since there is no route change. Then, we append `/hello` and `/hello/world` to the history, and then we move to the left to get `/hello`. This is followed by going to `/` which causes all right history to be erased. Hence, at the end we have `[/, /hello, /]` as the history.

### Route templates

Flet implements **route templates** which make it easy to match and parse route patterns:

In [3]:
def template_route_example(route: str):
    troute = ft.TemplateRoute(route)    # e.g. page.route (str)
    if troute.match("/books/:id"):
        print("book id:", troute.id)
    elif troute.match("/accounts/:account_id/orders/:order_id"):
        print("account id:", troute.account_id, "| order id:", troute.order_id)
    else:
        print("<Unknown route>")

template_route_example("/books/12345")
template_route_example("/accounts/USER/orders/11")
template_route_example("/unknown/route")

book id: 12345
account id: USER | order id: 11
<Unknown route>


### Page views

A page is not literally just a "single page" but a container for a list of views. 
On the bottom of the view stack is a **root view** (that cannot be popped), while the **top view** at the end of the list is that which is displayed. The views list represents navigator history. Page has `page.views` property to access views collection.

![Flet page is a collection of views with 🔙 to pop the topmost displayed view. [Source](https://flet.dev/img/docs/navigation-routing/page-views.svg) (docs)](img/flet/page-views.svg)

From the above subsections, we know how to extract routes from URL. 
How do we match these to specific views? To do this, we create `ft.View` objects
which take in a `route` attribute. Since only the top-most view is displayed what
we can do is clear the views list and just add the view that we want to show. 
This means that the views list is always a singleton.

```python
import asyncio
import flet as ft

HOME_ROUTE = "/"
STORE_ROUTE = "/store"
ORDERS_ROUTE = "/orders"

home_view = lambda page: ft.View(
    route=HOME_ROUTE,
    controls=[
        ft.AppBar(title=ft.Text("Home"), bgcolor=ft.Colors.SURFACE_CONTAINER_HIGHEST),
        ft.Button("Visit Store", on_click=lambda e: asyncio.create_task(page.push_route(STORE_ROUTE))),
        ft.Button("Check Orders", on_click=lambda e: asyncio.create_task(page.push_route(ORDERS_ROUTE))),
    ],
)

store_view = lambda page: ft.View(
    route=STORE_ROUTE,
    controls=[
        ft.AppBar(title=ft.Text("Store"), bgcolor=ft.Colors.SURFACE_CONTAINER_HIGHEST),
        ft.Button("Go Home", on_click=lambda e: asyncio.create_task(page.push_route(HOME_ROUTE))),
    ],
)

orders_view = lambda page: ft.View(
    route=ORDERS_ROUTE,
    controls=[
        ft.AppBar(title=ft.Text("Orders"), bgcolor=ft.Colors.SURFACE_CONTAINER_HIGHEST),
        ft.Button("Go Home", on_click=lambda e: asyncio.create_task(page.push_route(HOME_ROUTE))),
    ],
)


async def main(page: ft.Page):
    def route_change(e):
        routes_map = {
            HOME_ROUTE: home_view,
            STORE_ROUTE: store_view,
            ORDERS_ROUTE: orders_view
        }

        page.views.clear()  # flush prev view
        page.views.append(routes_map.get(e.route, home_view)(page))
        page.update()

    page.on_route_change = route_change
    page.views.append(home_view(page))
    page.update()
    

if __name__ == "__main__":
    ft.run(main)
```

Hence, [URL routes]{.underline} are mapped to [view object]{.underline} which are pushed to the top of the [views list]{.underline} resulting in it being displayed:

<video
  src="./img/flet/navigation-views.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

:::{.callout-note}
This does not take full advantage of the views feature. 
But the benefit of this approach is having clean URL routing where
browser 🡄, 🡆 works. Moreover, we never have to bother with checking for 
UI desync, i.e. ensuring that the URL is in 1-1 correspondence with the 
view that is showing in the UI.

:::

## Async apps

Flet app can be written as an async app and use `asyncio` and other Python async libraries. 
Async functions is naturally supported since Flet utilizes FastAPI as its underlying web server for running Flet applications. By default, Flet executes control [event handlers]{.underline} in [separate threads]{.underline} &mdash; this makes sense, but could be an ineffective usage of CPU, e.g. it does nothing while waiting for a HTTP response or executing `sleep()`. 

### Async event handlers

Note the use of `async` keyword:

```python
import datetime
import asyncio
import flet as ft

async def main(page: ft.Page):
    async def async_button(e):
        timestamp = datetime.datetime.now()
        await asyncio.sleep(3)
        page.add(ft.Text(f"Button clicked at {timestamp}"))
        page.update()

    page.add(ft.Button("async", on_click=async_button))
    page.update()


if __name__ == "__main__":
    ft.run(main)
```

For consecutive clicks this gives (i.e. **no blocking**, without async these will have 3-second gaps):

![](./img/flet/async.png)

:::{.callout-caution}
Note that there are no async lambdas, so you will have to write
async functions for event handlers. Also use `asyncio.sleep` to get non-blocking code.

::: 

### Background tasks

To run background tasks, we use `page.run_task(t)`. Moreover, [[background tasks must be async]{.mark}]{.underline}. Otherwise, the entire UI thread will block and the Flet app will freeze while waiting for the task to complete. Note that we can't use multiple threads since UI update in Flet is not thread-safe[^thread].

[^thread]: Flet requires all UI updates on the main thread, otherwise the UI may hang, crash, or behave inconsistently.

As a concrete example, let's implement a [countdown timer]{.underline}. This is like setting a countdown timer and immediately moving on to other things while it's running in the background. That is, you don't have to look at it all the time so your event loop is freed for you to focus on other tasks. Finally, you look at it again (1) once it completes and (2) your event loop gets to it (i.e. you're done with other tasks).

```python
import asyncio
import flet as ft

class Countdown(ft.Text):
    def __init__(self, seconds):
        super().__init__()
        self.seconds = seconds

    def did_mount(self):    # <1>
        self.running = True
        self.page.run_task(self.update_timer)

    def will_unmount(self): # <2>
        self.running = False 

    async def update_timer(self):
        while self.running:
            mins, secs = divmod(self.seconds, 60)
            self.value = "{:02d}:{:02d}".format(mins, secs)
            self.update()

            if self.seconds == 0:
                self.value += " 🛑" # <3>
                self.update()
                self.running = False
            else:
                await asyncio.sleep(1)
                self.seconds -= 1


def main(page: ft.Page):
    page.add(Countdown(10), Countdown(5))

if __name__ == "__main__":
    ft.run(main)
```

1. This uses [lifecycle methods](https://flet.dev/docs/getting-started/custom-controls/#life-cycle-methods) `did_mount` which is called after the control is added to the page. Then, we immediately run the countdown timer as background task.
2. This is called before the control is removed from the page. This method is overriden to execute clean-up code. Here cleanup is to set the `self.running` flag to `False` to terminate the `update_timer` loop.
3. Seeing 🛑 signifies the timer has stoped in the backend.

<video
  src="./img/flet/countdown.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

:::{.callout-note} 
Any interaction with the UI such as refreshing does not affect the timer cadence.

:::

## Deployment as FastAPI

Flet implements a **FastAPI app** to run the app as a dynamic website. It uses Uvicorn web server, by default, to run the app, but any ASGI-compatible server can be used instead. It follows that both sync and async event handlers can be used. Note that for web apps with a high number of users, threads are a scarce resource. If your app is mostly using I/O (database, web API), and you are able to use async-ready libraries, then async handlers are recommended.

:::{.callout-tip}
For compute-heavy / long-running background tasks, you can offload backend computation to a separate REST endpoint which implements a distributed task queue (e.g. autoscaling workers), to increase service availability.

:::

### Uvicorn

It suffices to set `export_asgi_app=True`:

```{.python filename=src/asgi_app.py}
import flet as ft

def main(page: ft.Page):
    page.add(ft.Text("Hello ASGI!"))

app = ft.app(main, export_asgi_app=True)
```

This allows running the app with **Uvicorn** (here with 4 worker processes):

```bash
❯ uvicorn src.asgi_app:app --workers 4
warning: `uv run` is experimental and may change without warning
INFO:     Started server process [84465]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
```

![](./img/flet/hello-uvicorn.png)

Or a more robust setup via **Gunicorn** with Uvicorn workers:

```bash
gunicorn --bind 0.0.0.0:8000 \
    -k uvicorn.workers.UvicornWorker \
    -w 4 \
    src.asgi_app:app
```

:::{.callout-note}
Each Uvicorn worker process handles many concurrent requests, but only one CPU-bound (i.e. **blocking**) task at a time. Hence, four worker processes means 4 event loops for asynchronous requests, and therefore 4 blocking requests are able to run allowing for **true parallelism**. Gunicorn schedules requests by round robin between the Uvicorn workers. 

:::


### Deployment strategies

This means we can deploy this the same way we deploy any FastAPI app. For example, we can containerize the UI application and deploy it in services like ECS. For scaling, we can follow [similar strategies](https://tech.olx.com/scaling-recommendations-service-at-olx-db4548813e3a) to scale FastAPI applications. However, note that since this is a frontend application, so there may be specific things that we have to keep in mind:

- **Memory.** First, each user connecting to it opens a **websocket connection**. This connection is stateful and are managed by worker processes, where each worker runs an event loop that can handle ~100 concurrent user sessions simultaneously. Since UI elements are stored in session memory, the memory footprint scales with the number of active users. Memory-intensive components can therefore become a bottleneck. So our deployment needs to provision adequate RAM based on expected user concurrency and typical UI complexity. It's also good practice to perform session cleanup to avoid memory leaks.

- **Latency.** Every user action on UI sends a message to Flet app and the app sends updated UI back to user. Make sure your hosting provider has multiple data centers, so you can run your app closer to the majority of your users.

- **State.** In Flet, UI state is primarily tied to a server-side *session* rather than to individual UI rebuilds. When a client connects, a **session** is created and bound to a worker process; the `Page` object and all control instances live in memory inside that worker. A browser refresh does not necessarily rebuild the page or reset state: if the client reconnects within the session’s lifetime[^session_lifetime], Flet reattaches the new websocket connection to the existing session (by including the previous `session_id`), and the [server-side UI tree]{.underline} is reused. From the user’s perspective, the UI appears unchanged because no new `Page` is constructed and no application code is re-executed. 

A page is only rebuilt when a new session is created, e.g. previous session expires, the worker process restarts or crashes, or the client reconnects to a different worker. In single-worker deployments this distinction is often invisible, which can give the impression that refreshes are “sticky” by default. With multiple workers or load balancers, however, a reconnect may be routed to a different worker that does not hold the original session in memory. In that case, Flet creates a new session, re-runs the app entry point, and the UI state is reset unless the application explicitly persists and restores state externally (e.g., via a database or cache).


:::{.callout-tip}
In general, working distributed systems (e.g. multiple Uvicorn workers) requires non-trivial state handling
You would need a remote Redis DB or other to **persist critical state** in case the user gets moved to a different worker after a crash or a scaling event.

:::

[^session_lifetime]: 3600 seconds by default.

<!-- ### More details... FastAPI

See [docs](https://flet.dev/docs/publish/web/dynamic-website/#assets) for more details ([assets]{.underline}, [environmental variables]{.underline}, etc). Further integration patterns with FastAPI are possible: -->

<!--
:::{.callout-warning}
This setup runs multiple apps in one worker process, as such can overload the server. It might be useful though for small or internal use-cases like admin, platform, or monitoring dashboards. Moreover, tight coupling of the services can simplify infra setup, CI/CD configuration, and local dev experience.
:::
-->


<!-- ```{.python filename="src/fastapi_mount.py"}
from contextlib import asynccontextmanager

import flet as ft
import flet.fastapi as flet_fastapi
from fastapi import FastAPI

@asynccontextmanager
async def lifespan(app: FastAPI):
    flet_fastapi.app_manager.start()
    yield
    flet_fastapi.app_manager.shutdown()

async def ui(page: ft.Page):
    page.add(ft.Text("Hello, Flet!"))


app = FastAPI(lifespan=lifespan)
app.mount("/flet-app", flet_fastapi.app(ui))
``` -->

<!-- This mounts the Flet app in the `/flet-app` path of a separate FastAPI app. The `lifespan` handles the start and cleanup of the app when the server starts and shuts down. Run this using uvicorn same as above: -->

<!-- ![](./img/flet/fastapi-mount.png) -->

<!-- We can host **multiple** Flet apps in the [same domain]{.underline}. Run using uvicorn as above: -->

<!-- ```{.python filename=src/fastapi_multi.py}
import flet as ft
import flet.fastapi as flet_fastapi

def root_main(page: ft.Page):
    page.add(ft.Text("This is root app!"))

def sub_main(page: ft.Page):
    page.add(ft.Text("This is sub app!"))


app = flet_fastapi.FastAPI()
app.mount("/", flet_fastapi.app(root_main))
app.mount("/sub-app", flet_fastapi.app(sub_main))
``` -->

<!-- ![](./img/flet/fastapi-multi.png) -->

<!--

:::{.callout-caution}
Note the path should be `/sub-app/` (notice the training slash) since it becomes the subapp's root path.
:::

-->


■

## Capstone: Todo App

The main element of this app is a **task widget**. We will extend our previous [simple implementation](/courses/app-dev/01-flet.html#composite-controls) of this into a full, feature-rich application. The general idea is the same, every interactive control calls a hook that affects the app state, and consequently, the UI elements (e.g. visibility, focus) which we build imperatively in response to the state change.

### Initial version

Start by initializing the app with `uv run flet create`. This gives us an initial template:

```bash
$ tree .
.
├── README.md
├── pyproject.toml
└── src
    ├── assets
    │   ├── icon.png
    │   └── splash_android.png
    └── main.py

3 directories, 5 files
```

In a real project, you will first have to edit details in the `pyproject.toml` file and `README.md`. For our purposes, we will focus on the `main.py` file. We add the following code:

```{.python filename=src/v1.py}
import flet as ft

def main(page: ft.Page):
    def add_clicked(e):
        task_list.controls.append(ft.Checkbox(label=new_task.value))
        new_task.value = ""     # blank = show hint text again
        main_col.update()

    new_task = ft.TextField(
        hint_text="What needs to be done?", 
        expand=True, 
        on_submit=add_clicked   # ENTER triggers on_submit
    )
    add_button = ft.FloatingActionButton(
        icon=ft.Icons.ADD, 
        on_click=add_clicked
    )
    
    new_task_row = ft.Row(controls=[new_task, add_button])
    task_list = ft.Column()
    main_col = ft.Column(width=600, controls=[new_task_row, task_list])
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.add(main_col)


if __name__ == "__main__":
    ft.run(main)
```

Reading this backwards, we see that the main control is `main_col` which is centered horizontally. The main column contains the text field for adding a task followed by the task list. The task list is a column whose controls are appended with new task items that are implemented as **checkbox**.

<video
  src="./img/flet-todo/todo-v1.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

The main hook here is `add_clicked` on the ADD button. Moreover, pressing ENTER on the keyboard triggers the same hook. This appends a checkbox to the task list column, resets the text field to blank (which makes the text hint show), and finally updates the main control.

### Refactoring into a control subclass

Note that the app packages nicely as a single composite control with methods (e.g. the event handlers). Moreover the app manages its data as class attributes. The following behaves exactly as the first one:

```{.python filename=src/v2.py}
import flet as ft

class TodoApp(ft.Column):
    def __init__(self, width: int):
        super().__init__(width=width)
        self.new_task = ft.TextField(
            hint_text="What needs to be done?", 
            expand=True, 
            on_submit=self.add_clicked   # ENTER triggers on_submit
        )
        self.add_button = ft.FloatingActionButton(
            icon=ft.Icons.ADD, 
            on_click=self.add_clicked
        )
        self.new_task_row = ft.Row(controls=[self.new_task, self.add_button])
        self.task_list = ft.Column()
        self.controls.extend([self.new_task_row, self.task_list])

    def add_clicked(self, e):
        self.task_list.controls.append(ft.Checkbox(label=self.new_task.value))
        self.new_task.value = ""    # blank = show hint text again
        self.update()


def main(page: ft.Page):
    todo = TodoApp(width=600)
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.add(todo)


if __name__ == "__main__":
    ft.run(main)
```

This feels like a cleaner more maintainable implementation of the original app.

### Task items as composite controls

Each **task item** is a composite column control consisting of two views. The default visible view is `display_view` which is a row consisting of three controls: (1) a checkbox with the **task name** (`.label`) and a clickable **toggle** (the state can be accessed via the a boolean attribute `.value`), (2) an edit button which calls `edit_clicked`, and (3) a delete button which calls `delete_clicked`. Next, we have the `edit_view` which is similarly a row with `TextField` and a save button which calls `save_clicked`. 

![](./img/flet-todo/task_item.drawio.png)

^Actually the checkbox doesn't extend as in the figure. Otherwise, the ff. demo shows the buttons behavior:

<video
  src="./img/flet-todo/todo-v3.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

The definition of each hook can be seen in the code:

```{.python filename=src/v3.py}
import flet as ft
from typing import Callable

class TaskItem(ft.Column):
    def __init__(self, text: str, delete_hook: Callable):
        super().__init__()
        self.delete_hook = delete_hook
        self.checkbox = ft.Checkbox(label=text)
        
        # icons
        self.edit_icon = ft.IconButton(icon=ft.Icons.EDIT, on_click=self.edit_clicked)
        self.save_icon = ft.IconButton(icon=ft.Icons.SAVE, on_click=self.save_clicked)
        self.delete_icon = ft.IconButton(icon=ft.Icons.DELETE, on_click=self.delete_clicked)

        # views
        self.text_view = ft.Row([self.checkbox, self.edit_icon, self.delete_icon])
        self.text_edit = ft.TextField(self.checkbox.label, expand=True, on_submit=self.save_clicked)
        self.edit_view = ft.Row([self.text_edit, self.save_icon], visible=False)

        self.controls.extend([self.text_view, self.edit_view])

    def delete_clicked(self, e):
        """Remove this task from the todo list using external hook."""
        self.delete_hook(self)

    def edit_clicked(self, e):
        self.text_view.visible = False
        self.edit_view.visible = True
        self.update()

    def save_clicked(self, e):
        self.checkbox.label = self.text_edit.value
        self.text_view.visible = True
        self.edit_view.visible = False
        self.update()

    def is_isolated(self):
        return True
...
```

Observe that the event handlers explicitly flips the visibility of the two views so that only one view is visible at each time. Clicking edit, makes only the `edit_view` visible. This makes the edit field visible and clicking save does the reverse while replacing the checkbox label to the contents of the `TextField`. Finally, delete task signals the delete hook which calls an [external]{.underline} function which makes sense since its the outer app container which handles the list of `TaskItem`. Let us now proceed with the `TodoApp` which maintains this said list:

```{.python filename=src/v3.py}
...

class TodoApp(ft.Column):
    def __init__(self, page: ft.Page, width: int):
        super().__init__(width=width)
        self._page = page
        self.new_task = ft.TextField(
            hint_text="What needs to be done?", 
            expand=True, 
            on_submit=self.add_clicked   # ENTER triggers on_submit
        )
        self.add_button = ft.FloatingActionButton(
            icon=ft.Icons.ADD, 
            on_click=self.add_clicked
        )
        self.task_list = ft.Column()
        self.controls.extend([
            ft.Row(controls=[self.new_task, self.add_button]), 
            self.task_list
        ])

    def add_clicked(self, e):
        task = TaskItem(self.new_task.value, delete_hook=self.delete_task)
        self.task_list.controls.append(task)
        self.new_task.value = ""    # blank = show hint text again
        self.update()

    def is_isolated(self):
        return True
    
    def delete_task(self, task: TaskItem):
        dlg_modal = ft.AlertDialog(
            modal=True,
            title=ft.Text("Confirm delete"),
            content=ft.Text("Are you sure you want to delete this task?"),
            actions=[
                ft.Button(
                    "Yes", 
                    on_click=lambda e: (
                        self.task_list.controls.remove(task), 
                        self.update(), 
                        self._page.pop_dialog()
                    )
                ),
                ft.TextButton(
                    "No", 
                    on_click=lambda e: self._page.pop_dialog()
                ),
            ],
            actions_alignment=ft.MainAxisAlignment.END,
        )
        self._page.show_dialog(dlg_modal)
        self._page.update()
        

def main(page: ft.Page):
    todo = TodoApp(page, width=600)
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.add(ft.Text("Todo list 📝", size=50, weight=ft.FontWeight.BOLD), todo)

if __name__ == "__main__":
    ft.run(main)
```

You can see that the final app is simply the previous version a column containing a text field for new tasks and a column containing the task items. Hence, in this version, we added the task item functionality. 

Additionally, we have the `delete_task` function which opens a **delete modal** and removes a task from the list. Note that `.remove` is $O(n)$ but since the task list is <100 or <1000 at the very extreme (probably this has to be enforced in an actual app), it should be fine. The `delete_task` function is inserted into each `TaskItem` which calls `delete_task(self)` allowing the main `TodoApp` to know which task to delete from its list.

:::{.callout-note}
Here we used `TextButton` for "No" and `Button` for "Yes" in the delete modal. However, it's not apparent from the code that we only selected one over the other because of **styling** reasons &mdash; the latter is elevated while the former blends into the background. It might be better to make styling explicit and just use `Button`.

:::

### Final enhancements: active tasks, tab filters, focus

This final section from a UX perspective adds quality of life improvements to the application. But from the developer's point of view, the current code change versus the last section reflects how complex tracking state is with the imperative approach. In the next notebook, we will see how to implement this in a **declarative** way which may be a more sane way to handle state for complex applications.

<video
  src="./img/flet-todo/todo-v4.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

There is a lot to unpack here. First, we see that the app does **autofocus** on text fields, e.g. the new task text field when the app starts, and immediately when a new task is added. Moreover, it focuses on the text field during task item edit mode. Also, a scrollbar appears whenever the allotted vertical space is exceeded by the task list and **autoscrolls** whenever a new task is added. We also see a footer which prints the number of active tasks. This is decreased whenever a task item is toggled as completed. 

The **tab filters** allow us to view task items with active or completed statuses. Note that any change while in one of the tabs applies to when we're at other tabs. This seems trivial, but in an imperative approach, making sure the state is consistent along any trajectory into the application is something that you may have to spend a lot of time on. Finally, the clear completed button removes all completed tasks. This means the active count must not be affected which can be seen here.

```{.python filename=src/v4.py}
import flet as ft
from typing import Callable

class TaskItem(ft.Column):
    def __init__(self, text: str, status_hook: Callable, delete_hook: Callable):
        super().__init__()
        
        self.delete_hook = delete_hook
        self.checkbox = ft.Checkbox(label=text, on_change=status_hook)
        
        self.edit_icon = ft.IconButton(icon=ft.Icons.EDIT, on_click=self.edit_clicked)
        self.save_icon = ft.IconButton(icon=ft.Icons.SAVE, on_click=self.save_clicked)
        self.delete_icon = ft.IconButton(icon=ft.Icons.DELETE, on_click=self.delete_clicked)
        
        self.text_view = ft.Row([self.checkbox, self.edit_icon, self.delete_icon])
        self.text_edit = ft.TextField(self.checkbox.label, expand=True, on_submit=self.save_clicked)
        self.edit_view = ft.Row([self.text_edit, self.save_icon], visible=False)

        self.controls.extend([self.text_view, self.edit_view])

    def delete_clicked(self, e):
        """Remove this task from the todo list using external hook."""
        self.delete_hook(self)

    async def edit_clicked(self, e):
        self.text_view.visible = False
        self.edit_view.visible = True
        self.update()
        await self.text_edit.focus()

    def save_clicked(self, e):
        self.checkbox.label = self.text_edit.value
        self.text_view.visible = True
        self.edit_view.visible = False
        self.update()

    def is_isolated(self):
        return True
...
```

What is added here is a `status_hook` for when the task status changes and we're monitoring active count and accurate views within tab filters. Note that forcing updates is necessary since each `TaskItem` is [isolated](/courses/app-dev/01-flet.html#isolated-controls). Next, the `edit_clicked` function is now async which is needed for **focus**. OK, that's fine. Now let us look at the main app: 

```{.python filename=src/v4.py}
...

class TodoApp(ft.Column):
    TAB_ALL = "all"
    TAB_ACTIVE = "active"
    TAB_COMPLETED = "completed"
    
    def __init__(self, page: ft.Page, width: int):
        super().__init__(width=width)
        self._page = page
        
        self.new_task = ft.TextField(
            hint_text="What needs to be done?", 
            expand=True, 
            on_submit=self.add_clicked   # ENTER triggers on_submit
        )
        
        self.add_button = ft.FloatingActionButton(
            icon=ft.Icons.ADD, 
            on_click=self.add_clicked
        )
        
        self.task_list = ft.ListView(height=250, spacing=10, auto_scroll=True)

        self.filter = ft.Tabs(
            selected_index=0,
            length=3,
            on_change=self.tabs_changed,
            content=ft.TabBar(
                scrollable=False,
                tabs=[
                    ft.Tab(label=TodoApp.TAB_ALL), 
                    ft.Tab(label=TodoApp.TAB_ACTIVE), 
                    ft.Tab(label=TodoApp.TAB_COMPLETED)
                ],
            )
        )

        self.active_count = ft.Text(
            "0 active tasks left.", 
            color=ft.Colors.GREY_400
        )

        self.clear_completed = ft.Button(
            "Clear Completed", 
            on_click=lambda e: self.confirm_clear_completed(),
            style=ft.ButtonStyle(shape=ft.RoundedRectangleBorder(radius=10))
        )

        self.controls.extend([
            ft.Row(controls=[self.new_task, self.add_button]), 
            self.filter,
            self.task_list,
            ft.Divider(thickness=0.5, color=ft.Colors.GREY_600),
            ft.Row(
                controls=[self.active_count, self.clear_completed], 
                alignment=ft.MainAxisAlignment.SPACE_BETWEEN
            )
        ])

    async def add_clicked(self, e):
        task = TaskItem(
            self.new_task.value, 
            status_hook=self.on_status_change, 
            delete_hook=self.confirm_delete_task
        )
        self.task_list.controls.append(task)
        self.new_task.value = ""    # note: blank = show hint text again
        self.on_status_change()     # new task => reflect +1 to active count
        self.update()
        await self.new_task.focus()

    def is_isolated(self):
        return True
    
    def on_status_change(self):
        num_active = sum(1 for task in self.task_list.controls if not self.is_completed(task))
        self.active_count.value = f"{num_active} active tasks left."
        self.update()

    def confirm_delete_task(self, task):
        delete_dialog = ft.AlertDialog(
            modal=True,
            title=ft.Text("Confirm delete"),
            content=ft.Text("Are you sure you want to delete this task?"),
            actions=[
                ft.Button(
                    "Yes", 
                    on_click=lambda e: (
                        self.delete_task(task), 
                        self.update(), 
                        self._page.pop_dialog()
                    )
                ),
                ft.TextButton(
                    "No", 
                    on_click=lambda e: self._page.pop_dialog()
                ),
            ],
            on_dismiss=lambda e: self.on_status_change(),
            actions_alignment=ft.MainAxisAlignment.END,
        )
        self._page.show_dialog(delete_dialog)
        self._page.update()
    
    def delete_task(self, task: TaskItem):
        self.task_list.controls.remove(task)
        
    def confirm_clear_completed(self):
        delete_dialog = ft.AlertDialog(
            modal=True,
            title=ft.Text("Confirm delete"),
            content=ft.Text("Are you sure you want to delete completed tasks?"),
            actions=[
                ft.Button(
                    "Yes", 
                    on_click=lambda e: (
                        self.delete_completed_tasks(),
                        self.update(), 
                        self._page.pop_dialog()
                    )
                ),
                ft.TextButton(
                    "No", 
                    on_click=lambda e: self._page.pop_dialog()
                ),
            ],
            on_dismiss=lambda e: self.on_status_change(),
            actions_alignment=ft.MainAxisAlignment.END,
        )
        self._page.show_dialog(delete_dialog)
        self._page.update()

    def delete_completed_tasks(self):
        for task in self.task_list.controls[:]:
            if self.is_completed(task):
                self.task_list.controls.remove(task)
    
    def tabs_changed(self, e):
        self.update()

    def is_completed(self, task: TaskItem):
        return task.checkbox.value

    def before_update(self):
        visible_fn = {
            TodoApp.TAB_ALL: lambda task: True,
            TodoApp.TAB_ACTIVE: lambda task: not self.is_completed(task),
            TodoApp.TAB_COMPLETED: lambda task: self.is_completed(task),
        }
        selected_idx = self.filter.selected_index
        selected_tab = self.filter.content.tabs[selected_idx].label
        for task in self.task_list.controls:
            task.visible = visible_fn[selected_tab](task)


async def main(page: ft.Page):
    todo = TodoApp(page, width=600)
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.add(ft.Text("Todo list 📝", size=50, weight=ft.FontWeight.BOLD), todo)
    await todo.new_task.focus()

if __name__ == "__main__":
    ft.run(main)
```

Again reading backward, we see that the app starts by focusing on the `new_task` field. This is refocused each time a new task is added as can be seen in the `await self.new_task.focus()` line. The `todo` app is a composite column control that consists of the new task field, followed by the tab filters, the task list, and the footer. Note that the add task hook `add_clicked` triggers `self.on_status_change()` which rebuilds the footer with the new count (i.e. +1):

```python
def on_status_change(self):
    num_active = sum(1 for task in self.task_list.controls if not self.is_completed(task))
    self.active_count.value = f"{num_active} active tasks left."
    self.update()
```

Again, the new task field is a row consisting of a `TextField` and a `FloatingActionButton` with an ADD icon. Autoscroll for the task list turns out to be implemented in the Flet library, hence we simply do:

```python
self.task_list = ft.ListView(height=250, spacing=10, auto_scroll=True)
```

Managing the behavior w.r.t. tab filters is a bit more involved. First, it is defined as follows:

```python
TAB_ALL = "all"
TAB_ACTIVE = "active"
TAB_COMPLETED = "completed"

self.filter = ft.Tabs(
    selected_index=0,
    length=3,
    on_change=self.tabs_changed,
    content=ft.TabBar(
        scrollable=False,
        tabs=[
            ft.Tab(label=TodoApp.TAB_ALL), 
            ft.Tab(label=TodoApp.TAB_ACTIVE), 
            ft.Tab(label=TodoApp.TAB_COMPLETED)
        ],
    )
)
```

Hence, it starts with "all", then "active" in the middle, and "completed" last. To actually trigger the effect of these in the UI, a hook is triggered `self.tabs_changed` which actually only does a `.update()` of the `todo` app. The trick is to use [lifecycle methods](https://docs.flet.dev/cookbook/custom-controls/#life-cycle-methods). Notice that by doing CTRL+F, you will not find a usage of `before_update`. This turns out to be a lifecycle method that is run each time the component is updated. 

Since `self.update()` is performed within `tabs_changed` in `todo` when a new tab is selected, this then runs `before_update`. Moreover, this is done whenever any other part of the app updates (e.g. a task item toggles status, then `on_status_change` is triggered which does an update, so that `before_update` is also inserted as an in between). The lifecycle method simply updates the visibility of the tasks depending on the tab selected:

```python
def before_update(self):    # lifecycle method! self = todo
    visible_fn = {
        TodoApp.TAB_ALL: lambda task: True,
        TodoApp.TAB_ACTIVE: lambda task: not self.is_completed(task),
        TodoApp.TAB_COMPLETED: lambda task: self.is_completed(task),
    }
    selected_idx = self.filter.selected_index
    selected_tab = self.filter.content.tabs[selected_idx].label
    for task in self.task_list.controls:
        task.visible = visible_fn[selected_tab](task)
```

:::{.callout-warning}
`before_update()` method is called every time when the control is being updated. Make sure not to call `update()` method within `before_update()`.

:::

Finally, we have a button for clearing completed tasks. This simply renders a dialog modal that calls a `delete_completed_tasks` which iterates over completed tasks and removes them from the task list. A subtle detail is that we iterate over a copy `self.task_list.controls[:]` since removing modifies the list (little Python gotcha).

## Final remarks 

The final code looks clear and straightforward. But this is only after taking a lot of experimentation &mdash; to figure out, for example, the minimal number of event handlers and the simplest patterns needed to maintain the consistency between UI components and actual application state (no. of active tasks, visible tasks). In particular, it takes added cognitive load to determine where to put page & component updates. 

In general, you constantly have to **synchronize** the app state and every UI element that depends on it. As an app grows, the number of places that must stay synchronized grows exponentially. In the next notebook, we explore a **declarative approach** to building Flet applications, where state and UI are clearly separated, and a single source of truth drives all UI updates: $\text{UI} = f(\text{state})$.